# Analisis de Datos Avanzados

Este notebook consolida la exploración de tipos nativos avanzados de PostgreSQL y el análisis de extensiones aplicables a Ecommify a partir del dataset Olist.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

data_path = Path.cwd()

csv_files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

datasets = {name: pd.read_csv(data_path / file_name) for name, file_name in csv_files.items()}

df_customers = datasets["customers"]
df_geolocation = datasets["geolocation"]
df_orders = datasets["orders"]
df_order_items = datasets["order_items"]
df_order_payments = datasets["order_payments"]
df_order_reviews = datasets["order_reviews"]
df_products = datasets["products"]
df_sellers = datasets["sellers"]
df_category_translation = datasets["category_translation"]

print("Datasets cargados para analisis avanzado:")
for name, df in datasets.items():
    print(f"- {name}: {df.shape[0]:,} filas x {df.shape[1]} columnas")

Datasets cargados para analisis avanzado:
- customers: 99,441 filas x 5 columnas
- geolocation: 1,000,163 filas x 5 columnas
- orders: 99,441 filas x 8 columnas
- order_items: 112,650 filas x 7 columnas
- order_payments: 103,886 filas x 5 columnas
- order_reviews: 99,224 filas x 7 columnas
- products: 32,951 filas x 9 columnas
- sellers: 3,095 filas x 4 columnas
- category_translation: 71 filas x 2 columnas


## Exploracion de datos nativos

En esta sección se preparan estructuras derivadas del dataset Olist para explorar cómo modelarlas con tipos nativos avanzados de PostgreSQL. El objetivo no es reemplazar el modelo relacional base, sino identificar atributos semi-estructurados o derivados que podrían beneficiarse de `JSONB`, `TEXT[]` y `TSTZRANGE`.

### Objetivos

- Preparar datos del dataset Olist en Colab para escenarios con tipos avanzados.
- Transformar datos relacionales a estructuras aptas para PostgreSQL avanzado.
- Construir ejemplos concretos de transformación:
  - `product_specifications` -> `JSONB`
  - `product_photos` -> `TEXT[]`
  - `promotion_period` -> `TSTZRANGE`
- Diseñar consultas que aprovechen estos tipos.
- Justificar técnicamente qué campo usar, con qué tipo y por qué.

> Nota: algunos campos, como `product_photos` y `promotion_period`, se construyen como atributos derivados de exploración porque el dataset Olist no incluye URLs reales de fotos ni una tabla nativa de promociones.

In [2]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# 1. Preparacion de estructuras derivadas para PostgreSQL avanzado
native_products = df_products.copy()

# JSONB: empaquetar especificaciones fisicas y metadatos descriptivos del producto
native_products["product_specifications"] = native_products.apply(
    lambda row: {
        "category": row["product_category_name"],
        "name_length": None if pd.isna(row["product_name_lenght"]) else int(row["product_name_lenght"]),
        "description_length": None if pd.isna(row["product_description_lenght"]) else int(row["product_description_lenght"]),
        "photos_qty": None if pd.isna(row["product_photos_qty"]) else int(row["product_photos_qty"]),
        "dimensions_cm": {
            "length": None if pd.isna(row["product_length_cm"]) else float(row["product_length_cm"]),
            "height": None if pd.isna(row["product_height_cm"]) else float(row["product_height_cm"]),
            "width": None if pd.isna(row["product_width_cm"]) else float(row["product_width_cm"])
        },
        "weight_g": None if pd.isna(row["product_weight_g"]) else float(row["product_weight_g"])
    },
    axis=1
)

# ARRAY TEXT[]: el dataset solo trae cantidad de fotos; se generan identificadores sinteticos para ilustrar modelado en arrays
native_products["product_photos"] = native_products.apply(
    lambda row: [] if pd.isna(row["product_photos_qty"]) or int(row["product_photos_qty"]) <= 0
    else [f"photo_{idx + 1}" for idx in range(int(row["product_photos_qty"]))],
    axis=1
)

# TSTZRANGE: atributo derivado de ejemplo para periodos promocionales por categoria
category_base = (
    native_products[["product_id", "product_category_name"]]
    .dropna(subset=["product_category_name"])
    .head(12)
    .reset_index(drop=True)
)
category_base["promotion_start"] = pd.to_datetime("2018-01-01") + pd.to_timedelta(category_base.index * 7, unit="D")
category_base["promotion_end"] = category_base["promotion_start"] + pd.to_timedelta(14, unit="D")
category_base["promotion_period"] = category_base.apply(
    lambda row: f"tstzrange('{row['promotion_start'].isoformat()}', '{row['promotion_end'].isoformat()}', '[)')",
    axis=1
)

native_type_justification = pd.DataFrame([
    {
        "campo_propuesto": "product_specifications",
        "origen": "products",
        "tipo_postgresql": "JSONB",
        "motivo": "Agrupa atributos de producto con estructura jerarquica, facilita consultas por clave y soporta evolucion del esquema sin romper el modelo canonico."
    },
    {
        "campo_propuesto": "product_photos",
        "origen": "products.product_photos_qty",
        "tipo_postgresql": "TEXT[]",
        "motivo": "Permite representar un conjunto acotado y ordenado de referencias de fotos cuando el caso de uso es lectura directa del producto."
    },
    {
        "campo_propuesto": "promotion_period",
        "origen": "atributo derivado para campanas/promociones",
        "tipo_postgresql": "TSTZRANGE",
        "motivo": "Modela intervalos temporales con operadores nativos de superposicion, contencion y vigencia para promociones activas."
    }
])

print("MUESTRA DE PRODUCTOS CON TIPOS NATIVOS AVANZADOS")
display(
    native_products[["product_id", "product_category_name", "product_specifications", "product_photos"]]
    .head(5)
)

print("\nTABLA DE JUSTIFICACION DE TIPOS AVANZADOS")
display(native_type_justification)

print("\nMUESTRA DE PERIODOS PROMOCIONALES DERIVADOS")
display(category_base[["product_id", "product_category_name", "promotion_start", "promotion_end", "promotion_period"]].head(8))

MUESTRA DE PRODUCTOS CON TIPOS NATIVOS AVANZADOS


,product_id,product_category_name,product_specifications,product_photos
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,"{'category': 'perfumaria', 'name_length': 40, ...",[photo_1]
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,"{'category': 'artes', 'name_length': 44, 'desc...",[photo_1]
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,"{'category': 'esporte_lazer', 'name_length': 4...",[photo_1]
3,cef67bcfe19066a932b7673e239eb23d,bebes,"{'category': 'bebes', 'name_length': 27, 'desc...",[photo_1]
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,"{'category': 'utilidades_domesticas', 'name_le...","[photo_1, photo_2, photo_3, photo_4]"



TABLA DE JUSTIFICACION DE TIPOS AVANZADOS


,campo_propuesto,origen,tipo_postgresql,motivo
0,product_specifications,products,JSONB,Agrupa atributos de producto con estructura je...
1,product_photos,products.product_photos_qty,TEXT[],Permite representar un conjunto acotado y orde...
2,promotion_period,atributo derivado para campanas/promociones,TSTZRANGE,Modela intervalos temporales con operadores na...



MUESTRA DE PERIODOS PROMOCIONALES DERIVADOS


,product_id,product_category_name,promotion_start,promotion_end,promotion_period
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,2018-01-01,2018-01-15,"tstzrange('2018-01-01T00:00:00', '2018-01-15T0..."
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,2018-01-08,2018-01-22,"tstzrange('2018-01-08T00:00:00', '2018-01-22T0..."
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,2018-01-15,2018-01-29,"tstzrange('2018-01-15T00:00:00', '2018-01-29T0..."
3,cef67bcfe19066a932b7673e239eb23d,bebes,2018-01-22,2018-02-05,"tstzrange('2018-01-22T00:00:00', '2018-02-05T0..."
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,2018-01-29,2018-02-12,"tstzrange('2018-01-29T00:00:00', '2018-02-12T0..."
5,41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,2018-02-05,2018-02-19,"tstzrange('2018-02-05T00:00:00', '2018-02-19T0..."
6,732bd381ad09e530fe0a5f457d81becb,cool_stuff,2018-02-12,2018-02-26,"tstzrange('2018-02-12T00:00:00', '2018-02-26T0..."
7,2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,2018-02-19,2018-03-05,"tstzrange('2018-02-19T00:00:00', '2018-03-05T0..."


In [3]:
native_sql_examples = {
    "ddl": """
CREATE TABLE products_native (
    product_id VARCHAR(32) PRIMARY KEY,
    product_category_name VARCHAR(120),
    product_specifications JSONB,
    product_photos TEXT[]
);

CREATE TABLE product_promotions (
    product_id VARCHAR(32) PRIMARY KEY,
    promotion_period TSTZRANGE NOT NULL
);
""",
    "jsonb_query": """
SELECT
    product_id,
    product_specifications ->> 'category' AS category,
    (product_specifications -> 'dimensions_cm' ->> 'length')::numeric AS length_cm
FROM products_native
WHERE product_specifications @> '{"category": "beleza_saude"}'::jsonb;
""",
    "array_query": """
SELECT
    product_id,
    cardinality(product_photos) AS total_photos,
    product_photos[1] AS first_photo
FROM products_native
WHERE cardinality(product_photos) >= 3;
""",
    "range_query": """
SELECT
    product_id,
    promotion_period
FROM product_promotions
WHERE promotion_period @> TIMESTAMPTZ '2018-02-10 12:00:00+00';
""",
    "combined_query": """
SELECT
    pn.product_id,
    pn.product_category_name,
    cardinality(pn.product_photos) AS photos_count,
    pp.promotion_period
FROM products_native pn
JOIN product_promotions pp
    ON pp.product_id = pn.product_id
WHERE pn.product_specifications ? 'dimensions_cm'
  AND cardinality(pn.product_photos) > 0
  AND pp.promotion_period && TSTZRANGE(
        TIMESTAMPTZ '2018-02-01 00:00:00+00',
        TIMESTAMPTZ '2018-02-20 00:00:00+00',
        '[)'
      );
"""
}

print("DDL PROPUESTO PARA TIPOS NATIVOS")
print(native_sql_examples["ddl"])

print("CONSULTA JSONB")
print(native_sql_examples["jsonb_query"])

print("CONSULTA ARRAY TEXT[]")
print(native_sql_examples["array_query"])

print("CONSULTA TSTZRANGE")
print(native_sql_examples["range_query"])

print("CONSULTA COMBINADA")
print(native_sql_examples["combined_query"])

native_index_notes = pd.DataFrame([
    {"tipo": "JSONB", "indice_sugerido": "GIN sobre product_specifications", "beneficio": "Filtrado por claves y contencion de documentos"},
    {"tipo": "TEXT[]", "indice_sugerido": "GIN sobre product_photos", "beneficio": "Busqueda de elementos dentro del arreglo"},
    {"tipo": "TSTZRANGE", "indice_sugerido": "GiST sobre promotion_period", "beneficio": "Consultas de contencion y solapamiento temporal"}
])

print("INDICES SUGERIDOS PARA APROVECHAR TIPOS NATIVOS")
display(native_index_notes)

DDL PROPUESTO PARA TIPOS NATIVOS

CREATE TABLE products_native (
    product_id VARCHAR(32) PRIMARY KEY,
    product_category_name VARCHAR(120),
    product_specifications JSONB,
    product_photos TEXT[]
);

CREATE TABLE product_promotions (
    product_id VARCHAR(32) PRIMARY KEY,
    promotion_period TSTZRANGE NOT NULL
);

CONSULTA JSONB

SELECT
    product_id,
    product_specifications ->> 'category' AS category,
    (product_specifications -> 'dimensions_cm' ->> 'length')::numeric AS length_cm
FROM products_native
WHERE product_specifications @> '{"category": "beleza_saude"}'::jsonb;

CONSULTA ARRAY TEXT[]

SELECT
    product_id,
    cardinality(product_photos) AS total_photos,
    product_photos[1] AS first_photo
FROM products_native
WHERE cardinality(product_photos) >= 3;

CONSULTA TSTZRANGE

SELECT
    product_id,
    promotion_period
FROM product_promotions
WHERE promotion_period @> TIMESTAMPTZ '2018-02-10 12:00:00+00';

CONSULTA COMBINADA

SELECT
    pn.product_id,
    pn.pro

,tipo,indice_sugerido,beneficio
0,JSONB,GIN sobre product_specifications,Filtrado por claves y contencion de documentos
1,TEXT[],GIN sobre product_photos,Busqueda de elementos dentro del arreglo
2,TSTZRANGE,GiST sobre promotion_period,Consultas de contencion y solapamiento temporal


## Analisis de extensiones (PostGIS, pg_trgm, hstore, pgcrypto)

Esta sección evalúa la aplicabilidad de extensiones comunes de PostgreSQL para Ecommify, considerando su valor funcional, costo operativo y alineación con los datos ya explorados.

### Objetivos

- Evaluar un caso de uso concreto por extensión dentro de Ecommify.
- Determinar si la extensión debe adoptarse ahora, más adelante o descartarse.
- Aterrizar el caso principal de PostGIS para optimizar costos de envío según distancia vendedor-cliente.
- Vincular el análisis del notebook con el documento final `Extensiones_PostgreSQL_Ecommify.md`.

In [4]:
extensions_analysis = pd.DataFrame([
    {
        "extension": "PostGIS",
        "caso_de_uso": "Calculo de distancia vendedor-cliente para estimar costos y SLA de envio",
        "aplicabilidad": "Alta",
        "decision": "Adoptar en fase de optimizacion geografica",
        "justificacion": "El dataset contiene prefijos postales y coordenadas depurables; habilita ST_Distance, buffers y analitica espacial con mejor precision que calculos manuales."
    },
    {
        "extension": "pg_trgm",
        "caso_de_uso": "Busqueda de productos tolerante a errores tipograficos en nombres y categorias",
        "aplicabilidad": "Alta",
        "decision": "Adoptar para catalogo y buscador",
        "justificacion": "Mejora la experiencia de busqueda cuando el usuario escribe con errores, abreviaciones o diferencias ortograficas."
    },
    {
        "extension": "hstore",
        "caso_de_uso": "Bolsa ligera de atributos variables en productos o eventos",
        "aplicabilidad": "Media-Baja",
        "decision": "No priorizar; preferir JSONB",
        "justificacion": "El proyecto se beneficia mas de JSONB por jerarquia, operadores mas ricos e indexacion flexible; hstore solo seria util en key-value simple y plano."
    },
    {
        "extension": "pgcrypto",
        "caso_de_uso": "Hash o seudonimizacion de identificadores sensibles y generacion de tokens",
        "aplicabilidad": "Media",
        "decision": "Adoptar si se publican datos o se requiere proteccion de PII",
        "justificacion": "Aporta hashing y funciones criptograficas para proteger identificadores antes de exponer datasets o analitica compartida."
    }
])

print("MATRIZ DE APLICABILIDAD DE EXTENSIONES")
display(extensions_analysis)

postgis_example = """
-- Ejemplo: costo de envio basado en distancia geografica aproximada
SELECT
    oi.order_id,
    oi.seller_id,
    c.customer_id,
    ST_Distance(
        ST_SetSRID(ST_MakePoint(gs.geolocation_lng, gs.geolocation_lat), 4326)::geography,
        ST_SetSRID(ST_MakePoint(gc.geolocation_lng, gc.geolocation_lat), 4326)::geography
    ) / 1000 AS distancia_km
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN customers c ON c.customer_id = o.customer_id
JOIN sellers s ON s.seller_id = oi.seller_id
JOIN geolocation_reference gs ON gs.zip_code_prefix = s.seller_zip_code_prefix
JOIN geolocation_reference gc ON gc.zip_code_prefix = c.customer_zip_code_prefix;
"""

pg_trgm_example = """
-- Ejemplo: busqueda tolerante a errores tipograficos
SELECT
    product_id,
    product_category_name,
    similarity(product_category_name, 'beleza e saude') AS score
FROM products
WHERE product_category_name % 'beleza e saude'
ORDER BY score DESC;
"""

pgcrypto_example = """
-- Ejemplo: seudonimizacion de identificadores de cliente
SELECT
    customer_id,
    encode(digest(customer_unique_id, 'sha256'), 'hex') AS customer_hash
FROM customers;
"""

print("EJEMPLO POSTGIS")
print(postgis_example)

print("EJEMPLO PG_TRGM")
print(pg_trgm_example)

print("EJEMPLO PGCRYPTO")
print(pgcrypto_example)

final_extension_decision = pd.DataFrame([
    {"extension": "PostGIS", "decision_final": "Si", "prioridad": "Alta", "motivo": "Entrega valor directo en optimizacion logistica y analitica geografica."},
    {"extension": "pg_trgm", "decision_final": "Si", "prioridad": "Alta", "motivo": "Mejora la experiencia de busqueda y recuperacion aproximada de productos."},
    {"extension": "hstore", "decision_final": "No por ahora", "prioridad": "Baja", "motivo": "JSONB cubre mejor los requerimientos del dominio con mayor flexibilidad."},
    {"extension": "pgcrypto", "decision_final": "Condicional", "prioridad": "Media", "motivo": "Relevante si se expone informacion sensible o se comparte analitica con datos protegidos."}
])

print("DECISION FINAL DE EXTENSIONES")
display(final_extension_decision)

MATRIZ DE APLICABILIDAD DE EXTENSIONES


,extension,caso_de_uso,aplicabilidad,decision,justificacion
0,PostGIS,Calculo de distancia vendedor-cliente para est...,Alta,Adoptar en fase de optimizacion geografica,El dataset contiene prefijos postales y coorde...
1,pg_trgm,Busqueda de productos tolerante a errores tipo...,Alta,Adoptar para catalogo y buscador,Mejora la experiencia de busqueda cuando el us...
2,hstore,Bolsa ligera de atributos variables en product...,Media-Baja,No priorizar; preferir JSONB,El proyecto se beneficia mas de JSONB por jera...
3,pgcrypto,Hash o seudonimizacion de identificadores sens...,Media,Adoptar si se publican datos o se requiere pro...,Aporta hashing y funciones criptograficas para...


EJEMPLO POSTGIS

-- Ejemplo: costo de envio basado en distancia geografica aproximada
SELECT
    oi.order_id,
    oi.seller_id,
    c.customer_id,
    ST_Distance(
        ST_SetSRID(ST_MakePoint(gs.geolocation_lng, gs.geolocation_lat), 4326)::geography,
        ST_SetSRID(ST_MakePoint(gc.geolocation_lng, gc.geolocation_lat), 4326)::geography
    ) / 1000 AS distancia_km
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN customers c ON c.customer_id = o.customer_id
JOIN sellers s ON s.seller_id = oi.seller_id
JOIN geolocation_reference gs ON gs.zip_code_prefix = s.seller_zip_code_prefix
JOIN geolocation_reference gc ON gc.zip_code_prefix = c.customer_zip_code_prefix;

EJEMPLO PG_TRGM

-- Ejemplo: busqueda tolerante a errores tipograficos
SELECT
    product_id,
    product_category_name,
    similarity(product_category_name, 'beleza e saude') AS score
FROM products
WHERE product_category_name % 'beleza e saude'
ORDER BY score DESC;

EJEMPLO PGCRYPTO

-- Ejemplo: seudonim

,extension,decision_final,prioridad,motivo
0,PostGIS,Si,Alta,Entrega valor directo en optimizacion logistic...
1,pg_trgm,Si,Alta,Mejora la experiencia de busqueda y recuperaci...
2,hstore,No por ahora,Baja,JSONB cubre mejor los requerimientos del domin...
3,pgcrypto,Condicional,Media,Relevante si se expone informacion sensible o ...
